In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from scipy.interpolate import RBFInterpolator
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import xarray as xr

from project.utils.plotting import ProjectCustomExtent, despine

plt.rcParams["figure.dpi"] = 150

In [ ]:

def plot_study_progress(study: optuna.Study, ax: mpl.axes.Axes):
    for trial in study.get_trials():
        if trial.number == study.best_trial.number:
            args = dict(
                lw = 1.6,
                color = 'k',
                zorder = 3,
                label='Best trial',
            )
        elif trial.state == 1:
            args = dict(
                lw = 0.8,
                color = 'tab:blue',
                zorder = 0,
                label = 'Complete trials',
            )
        elif trial.state == 2:
            args = dict(
                lw = 0.8,
                color = 'tab:orange',
                zorder = 1,
                label = 'Pruned trials',
            )
        else:
            args = dict(
                lw = 0.8,
                color = 'tab:red',
                zorder = 2,
                label = 'Failed trials',
            )

        intermediate_values = pd.DataFrame(
            [(k, v) for k, v in trial.intermediate_values.items()], columns=['Step', 'Value']).set_index('Step')

        ax.plot(intermediate_values, alpha=0.8, **args)

        despine(ax)


def plot_perf_interaction(df: pd.DataFrame, params: list[str], values: str):

    vmin = np.min(df[values]) + 1e-6
    vmax = np.quantile(df[values], 0.98)

    fig, axes = plt.subplots(len(params), len(params), figsize=(12, 12), sharex='col', sharey='row')

    for i, p_i in enumerate(params):
        for j, p_j in enumerate(params):
            ax = axes[i, j]

            if i <= j:
                im = ax.scatter(df[p_j], df[p_i], c=df[values],
                    s=15, alpha=1.0, ec='k', lw=0.5, cmap='coolwarm_r',
                    vmin=vmin, vmax=vmax)
            else:
                # --------------------------------------------------------------
                # 1️⃣ Settings
                # --------------------------------------------------------------
                grid_res = 100                     # finer grid gives smoother contours
                # --------------------------------------------------------------

                # --------------------------------------------------------------
                # 2️⃣ Pull the three variables from the dataframe
                # --------------------------------------------------------------
                x = df[p_j].astype(float).values   # horizontal axis
                y = df[p_i].astype(float).values   # vertical axis
                x += 1e-6 * np.random.randn(len(x))  # jitter to avoid duplicates
                y += 1e-6 * np.random.randn(len(y))  # jitter to
                z = df['value'].astype(float).values # the quantity we want to interpolate

                def get_min_max(arr, extend_factor=0.2):
                    mn, mx = arr.min(), arr.max()

                    rng = mx - mn
                    mn_ext = mn - extend_factor * rng
                    mx_ext = mx + extend_factor * rng

                    return mn_ext, mx_ext

                # --------------------------------------------------------------
                # 5️⃣ Interpolate the scattered (x, y, z) onto the grid
                # --------------------------------------------------------------
                # `points` must be shape (N, 2); `values` shape (N,)
                points = np.column_stack((x, y))
                points_normalized = StandardScaler().fit_transform(points)

                # --------------------------------------------------------------
                # 4️⃣ Build a regular grid on which we will evaluate the interpolation
                # --------------------------------------------------------------
                xmin, xmax = get_min_max(points[:, 0])
                ymin, ymax = get_min_max(points[:, 1])
                xi = np.linspace(xmin, xmax, grid_res)
                yi = np.linspace(ymin, ymax, grid_res)
                XI, YI = np.meshgrid(xi, yi)

                xmin_norm, xmax_norm = get_min_max(points_normalized[:, 0])
                ymin_norm, ymax_norm = get_min_max(points_normalized[:, 1])
                xi_norm = np.linspace(xmin_norm, xmax_norm, grid_res)
                yi_norm = np.linspace(ymin_norm, ymax_norm, grid_res)
                XI_norm, YI_norm = np.meshgrid(xi_norm, yi_norm)

                # `griddata` returns NaNs where the convex hull of the input points does not cover.
                # We mask those later so the contour routine does not complain.
                # ZI = griddata(points, z, (XI, YI), method=method)
                rbf = RBFInterpolator(points_normalized, z, kernel='thin_plate_spline', epsilon=2)
                ZI = rbf(np.column_stack([XI_norm.ravel(), YI_norm.ravel()])).reshape(XI.shape)
                ZI = np.clip(ZI, vmin, vmax)  # clip to avoid extreme outliers messing up the colour scale

                # Filled contours (you can tune `levels` or let matplotlib pick them)
                cf = ax.contourf(XI, YI, ZI, levels=19, cmap='coolwarm_r', vmin=vmin, vmax=vmax)
                # Optional thin black lines for the iso‑levels
                ax.contour(XI, YI, ZI, levels=19, colors='k', linewidths=0.4, vmin=vmin, vmax=vmax)

                # --------------------------------------------------------------
                # 7️⃣ Overlay the raw points (optional – makes the plot easier to read)
                # --------------------------------------------------------------
                ax.scatter(x, y, facecolor='k', edgecolor='w',
                        s=15, alpha=1.0, lw=1, zorder=10)

                ax.set_xlim(xmin, xmax)
                ax.set_ylim(ymin, ymax)

                # --------------------------------------------------------------
                # 8️⃣ Store the contour collection so the colour‑bar can reference it later
                # --------------------------------------------------------------
                # (only needed if you want a separate colour‑bar for the interpolated field;
                # otherwise the existing colour‑bar for the upper‑triangle scatter stays)
                im = cf   # <-- reuse the variable name `im` that the colour‑bar code expects

            despine(ax)

            x_label = p_j.replace('params_model.', '').replace('_', '\n').capitalize()
            y_label = p_i.replace('params_model.', '').replace('_', '\n').capitalize()

            if j == 0:
                ax.set_ylabel(y_label, size=7)
            else:
                ax.set_ylabel('')
            if i == len(params) - 1:
                ax.set_xlabel(x_label, size=7)
            else:
                ax.set_xlabel('')

    # Add colorbar
    axins = axes[0, -1].inset_axes([1.0, 0.0, 0.05, 1.0])

    cbar = fig.colorbar(im, fraction=0.02, pad=0.04, label='Validation loss (log scale)', orientation='vertical', cax=axins)

    # cbar.ax.xaxis.set_minor_locator(mpl.ticker.LogLocator(subs=np.arange(2, 9) * 0.1))
    # cbar.ax.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
    cbar.ax.tick_params(which='major', labelsize=5)
    cbar.ax.tick_params(which='minor', labelsize=5)

    # colorbar label size
    cbar.ax.yaxis.label.set_size(7)

    # Align x and y axis labels.
    fig.align_ylabels(axes[:, 0])
    fig.align_xlabels(axes[-1, :])

    # Decrease tick label size.
    for ax in axes.flatten():
        ax.tick_params(axis='both', which='major', labelsize=7)
        ax.tick_params(axis='both', which='minor', labelsize=7)


    return fig, axes


# Pretraining

In [ ]:
study = optuna.load_study(
        study_name=None,
        storage=f'sqlite:////net/argon/landclim/kraftb/machflow_routing/logs/pretraining/tune/optuna.db')

fig, ax = plt.subplots(figsize=(4, 4))

plot_study_progress(study, ax)

handles, labels = ax.get_legend_handles_labels()
handles_new = []
labels_new = []
for h, l in zip(handles, labels):
    if l not in labels_new:
        labels_new.append(l)
        handles_new.append(h)
ax.legend(handles_new, labels_new, loc='upper right', frameon=False)
despine(ax)
ax.set_title(f'Pretraining on PREVAH')

ax.set_xlabel('Epoch')
ax.set_ylabel(f'Validation NSE (normalized)')


In [ ]:
df = study.trials_dataframe()
df = df[df.state.isin(['COMPLETE'])]
params = [col for col in df.columns if col.startswith('params_model')]

fig, axes = plot_perf_interaction(df, params, 'value')

fig.suptitle('Pretraining on PREVAH', y=0.92, fontsize=16)

# Lumped models

In [ ]:
models = ['lumped_random', 'lumped_easy', 'lumped_hard']

study_dict = {}

for model in models:
    model_type = model.split('_')[0]
    split_type = model.split('_')[1]

    s = optuna.load_study(
        study_name=None,
        storage=f'sqlite:////net/argon/landclim/kraftb/machflow_routing/logs/{model_type}/{split_type}/tune/optuna.db')

    study_dict[model] = {'model_type': model_type, 'split_type': split_type, 'study': s}


fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=True, sharey='row')

for i, (key, value) in enumerate(study_dict.items()):
    model_type = value['model_type']
    split_type = value['split_type']
    study = value['study']

    ax = axes[i]

    plot_study_progress(study, ax)

    if i == 1:
        ax.set_xlabel('Epoch')

    ax.set_ylabel(f'Validation loss')

    ax.set_title(f'{model_type.capitalize()}-{split_type.capitalize()}')

ax = axes[0]
handles, labels = ax.get_legend_handles_labels()
handles_new = []
labels_new = []
for h, l in zip(handles, labels):
    if l not in labels_new:
        labels_new.append(l)
        handles_new.append(h)
ax.legend(handles_new, labels_new, loc='upper right', frameon=False)


In [ ]:
for key, value in study_dict.items():

    model_type = value['model_type']
    split_type = value['split_type']
    study = value['study']

    df = study.trials_dataframe()
    df = df[df.state.isin(['COMPLETE'])]
    params = [col for col in df.columns if col.startswith('params_model')]

    fig, axes = plot_perf_interaction(df, params, 'value')
    fig.suptitle(f'{model_type.capitalize()}-{split_type.capitalize()}', y=0.92, fontsize=16)

# Routed models

In [ ]:
models = ['routed_random', 'routed_easy', 'routed_hard']

study_dict = {}

for model in models:
    model_type = model.split('_')[0]
    split_type = model.split('_')[1]

    s = optuna.load_study(
        study_name=None,
        storage=f'sqlite:////net/argon/landclim/kraftb/machflow_routing/logs/{model_type}/{split_type}/tune/optuna.db')

    study_dict[model] = {'model_type': model_type, 'split_type': split_type, 'study': s}


fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=True, sharey='row')

for i, (key, value) in enumerate(study_dict.items()):
    model_type = value['model_type']
    split_type = value['split_type']
    study = value['study']

    ax = axes[i]

    plot_study_progress(study, ax)

    if i == 1:
        ax.set_xlabel('Epoch')

    ax.set_ylabel(f'Validation loss')

    ax.set_title(f'{model_type.capitalize()}-{split_type.capitalize()}')

ax = axes[0]
handles, labels = ax.get_legend_handles_labels()
handles_new = []
labels_new = []
for h, l in zip(handles, labels):
    if l not in labels_new:
        labels_new.append(l)
        handles_new.append(h)
ax.legend(handles_new, labels_new, loc='upper right', frameon=False)


In [ ]:
study_dict['routed_random']['study'].trials_dataframe()